# Определение ботов по событиям cookie

Для каждой cookie строю признаки по событиям её суточного окна и предсказываю вероятность положительного класса. В финальном решении 222 признака: 199 описывают поведение самой cookie, 23 — общие объявления с другими cookie. Модель — ансамбль CatBoost и LightGBM с весами 75/25, по три seed на каждый алгоритм.

Рядом лежат `data/train.csv`, `data/test.csv`, `data/events.csv.gz` и `sample_submission.csv`. В конце создаётся `submission.csv`.

Использован Python **3.11.9**. Зависимости: `numpy==2.2.6`, `pandas==2.2.3`, `scikit-learn==1.6.1`, `catboost==1.2.8`, `lightgbm==4.6.0`; установка — `pip install -r requirements.txt`. На macOS для LightGBM нужен OpenMP (`brew install libomp`). 


## 1. Данные и очистка


In [2]:
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
import platform
import random

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score

SEEDS = [42, 17, 123]
random.seed(42)
np.random.seed(42)
DATA_DIR = Path('data')
FEATURE_DIR = Path('solution_features')
FEATURE_DIR.mkdir(exist_ok=True)
print('Python:', platform.python_version())


Python: 3.11.9


In [3]:
DATE_COLUMNS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
STRING_COLUMNS = ['cookie_id', 'eid', 'event_name', 'platform', 'user_agent',
                  'item_id', 'item_category', 'item_location', 'seller_type', 'search_query']
train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'cookie_id': str}, parse_dates=DATE_COLUMNS)
test = pd.read_csv(DATA_DIR / 'test.csv', dtype={'cookie_id': str}, parse_dates=DATE_COLUMNS)
events = pd.read_csv(DATA_DIR / 'events.csv.gz', dtype={c: str for c in STRING_COLUMNS},
                     keep_default_na=False, parse_dates=['event_ts'])
for column in ['search_page', 'pointer_x', 'pointer_y']:
    events[column] = pd.to_numeric(events[column].replace('', np.nan), errors='raise')

assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert train.target.isin([0, 1]).all()
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
print(f'Train: {len(train):,}; test: {len(test):,}; events: {len(events):,}')
print(f'Доля ботов в train: {train.target.mean():.2%}')


Train: 11,091; test: 4,909; events: 328,905
Доля ботов в train: 8.11%


Из User-Agent выделяю семейство браузера, ОС и наличие признаков автоматизированного клиента. Версии браузеров и модели устройств не передаю в модель: они могут быстро меняться и описывать конкретный источник трафика.


In [4]:
EVENTS = ['search_results_view', 'item_view', 'photo_swipe', 'seller_page_view',
          'contact_phone_show', 'contact_chat_open', 'contact_message_sent',
          'favorite_add', 'login', 'captcha_shown']
CONTACTS = ['contact_phone_show', 'contact_chat_open', 'contact_message_sent']
PLATFORMS = ['web', 'android', 'ios', 'other', 'missing']
BROWSERS = ['chrome', 'firefox', 'safari', 'yandex', 'edge', 'headless',
            'avito_app', 'http_client', 'other', 'missing']
OS_NAMES = ['windows', 'macos', 'linux', 'android', 'ios', 'other', 'missing']


def normalize_platform(value):
    value = str(value).strip().lower()
    return {'desktop': 'web', 'web': 'web', 'android': 'android',
            'ios': 'ios', 'iphone': 'ios', '': 'missing'}.get(value, 'other')


@lru_cache(maxsize=4096)
def parse_ua(value):
    ua = str(value).strip().lower()
    if not ua:
        return 'missing', 'missing', False
    automated = any(token in ua for token in [
        'headless', 'python-requests', 'python-urllib', 'scrapy', 'curl/',
        'go-http-client', 'node-fetch', 'wget/', 'aiohttp', 'httpx/', 'selenium'])
    if 'avito/' in ua:
        browser = 'avito_app'
    elif 'headless' in ua:
        browser = 'headless'
    elif automated:
        browser = 'http_client'
    elif 'yabrowser' in ua:
        browser = 'yandex'
    elif 'edg/' in ua or 'edge/' in ua:
        browser = 'edge'
    elif 'firefox' in ua:
        browser = 'firefox'
    elif 'chrome' in ua or 'crios' in ua:
        browser = 'chrome'
    elif 'safari' in ua:
        browser = 'safari'
    else:
        browser = 'other'
    if 'android' in ua:
        os_name = 'android'
    elif 'iphone' in ua or 'ipad' in ua:
        os_name = 'ios'
    elif 'windows' in ua:
        os_name = 'windows'
    elif 'macintosh' in ua or 'mac os' in ua:
        os_name = 'macos'
    elif 'linux' in ua:
        os_name = 'linux'
    else:
        os_name = 'other'
    return browser, os_name, automated


Оставляю события из `[window_start_ts, window_end_ts)`, затем удаляю полные дубликаты исходных строк. Совпадения только по времени не считаю дублями. События сортирую внутри cookie по времени; метка `target` в построении признаков не участвует.


In [5]:
original_columns = events.columns.tolist()
events = events.merge(meta, on='cookie_id', how='left', validate='many_to_one', indicator=True)
assert events['_merge'].eq('both').all()
assert events.event_ts.notna().all()
inside = events.event_ts.ge(events.window_start_ts) & events.event_ts.lt(events.window_end_ts)
print('Событий вне окна:', int((~inside).sum()))
events = events.loc[inside].copy()
duplicates = events.duplicated(subset=original_columns)
print('Полных дублей внутри окна:', int(duplicates.sum()))
events = events.loc[~duplicates].copy()
assert events.event_ts.ge(events.cookie_created_at).all()
assert set(events.event_name).issubset(EVENTS)

events['platform_norm'] = events.platform.map(normalize_platform)
parsed = events.user_agent.map(parse_ua)
ua_columns = ['browser', 'os', 'ua_automation_hint']
events[ua_columns] = pd.DataFrame(parsed.tolist(), index=events.index, columns=ua_columns)
events = events.sort_values(['cookie_id', 'event_ts'], kind='stable').reset_index(drop=True)
print('Событий для признаков:', len(events))


Событий вне окна: 40779
Полных дублей внутри окна: 4293
Событий для признаков: 283833


## 2. Признаки поведения

Использую возраст cookie, состав действий, интервалы между событиями, плотность активности, сессии, разнообразие объектов и переходы между действиями. Координаты курсора рассматриваю как наблюдения при событиях, а не как непрерывную траекторию.

Для неопределённых отношений оставляю `NaN`: например, отсутствие просмотров не означает нулевую долю контактов на просмотр. Числовые пропуски обрабатываются самими моделями.

In [6]:
def ratio(a, b):
    return float(a / b) if b else np.nan


def entropy(values):
    counts = np.array(list(Counter(values).values()), dtype=float)
    if not len(counts):
        return np.nan
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def mode(values):
    counts = Counter(values)
    return min(counts, key=lambda x: (-counts[x], x)) if counts else 'missing'


def stats(values, extended=False):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    result = dict.fromkeys(['mean', 'std', 'median', 'max'], np.nan)
    if extended:
        result.update(dict.fromkeys(['min', 'p25', 'p75', 'p90'], np.nan))
    if len(x):
        result.update(mean=float(x.mean()), std=float(x.std(ddof=0)),
                      median=float(np.median(x)), max=float(x.max()))
        if extended:
            p25, p75, p90 = np.quantile(x, [.25, .75, .9])
            result.update(min=float(x.min()), p25=float(p25), p75=float(p75), p90=float(p90))
    return result


def add_stats(out, prefix, values, extended=False):
    out.update({f'{prefix}_{name}': value for name, value in stats(values, extended).items()})


def add_diversity(out, prefix, values):
    values = [v for v in values if v != '']
    counts = Counter(values)
    out.update({
        f'{prefix}_observed_count': len(values),
        f'{prefix}_nunique': len(counts),
        f'{prefix}_unique_fraction': ratio(len(counts), len(values)),
        f'{prefix}_entropy': entropy(values),
        f'{prefix}_top_share': ratio(max(counts.values(), default=0), len(values)),
    })


In [7]:
def activity_features(row, ev):
    n = len(ev)
    counts = Counter(ev.event_name)
    out = {'metadata__cookie_age_days': (row.window_end_ts - row.cookie_created_at).total_seconds() / 86400,
           'activity__event_count': n,
           'activity__event_type_nunique': len(counts),
           'activity__event_type_entropy': entropy(ev.event_name)}
    for name in EVENTS[:-1]:
        out[f'activity__{name}_count'] = counts[name]
        out[f'activity__{name}_share'] = ratio(counts[name], n)
    contacts = sum(counts[name] for name in CONTACTS)
    out['activity__contact_count'] = contacts
    out['activity__contact_share'] = ratio(contacts, n)
    for name, numerator, denominator in [
        ('contacts_per_item_view', contacts, counts['item_view']),
        ('photos_per_item_view', counts['photo_swipe'], counts['item_view']),
        ('favorites_per_item_view', counts['favorite_add'], counts['item_view']),
        ('seller_views_per_item_view', counts['seller_page_view'], counts['item_view']),
        ('item_views_per_search', counts['item_view'], counts['search_results_view']),
        ('messages_per_chat_open', counts['contact_message_sent'], counts['contact_chat_open']),
    ]:
        out[f'activity__{name}'] = ratio(numerator, denominator)
    return out


Интервалы и распределение событий по времени помогают различать регулярную автоматическую активность и более неравномерное поведение. Сессии разделяю паузой больше 30 минут.


In [8]:
def timing_features(row, ev):
    n = len(ev)
    t = (ev.event_ts - row.window_start_ts).dt.total_seconds().to_numpy()
    gaps = np.diff(t)
    duration = (row.window_end_ts - row.window_start_ts).total_seconds()
    out = {'timing__gap_count': len(gaps)}
    add_stats(out, 'timing__gap_seconds', gaps, extended=True)
    mean = float(gaps.mean()) if len(gaps) else np.nan
    std = float(gaps.std()) if len(gaps) else np.nan
    out.update({
        'timing__gap_cv': ratio(std, mean),
        'timing__gap_burstiness': ratio(std - mean, std + mean),
        'timing__gap_entropy': entropy(gaps),
        'timing__gap_unique_fraction': ratio(len(set(gaps)), len(gaps)),
        'timing__zero_gap_share': ratio(int((gaps == 0).sum()), len(gaps)),
    })
    for seconds in [1, 5, 30, 60, 300, 1800]:
        out[f'timing__gap_le_{seconds}s_share'] = ratio(int((gaps <= seconds).sum()), len(gaps))
    correlation = np.nan
    if len(gaps) >= 3 and gaps[:-1].std() > 0 and gaps[1:].std() > 0:
        correlation = float(np.corrcoef(gaps[:-1], gaps[1:])[0, 1])
    span = float(t[-1] - t[0]) if n else np.nan
    hours = ev.event_ts.dt.hour.to_numpy()
    out.update({
        'timing__gap_lag1_correlation': correlation,
        'timing__active_span_seconds': span,
        'timing__active_span_fraction': ratio(span, duration),
        'timing__first_event_offset_seconds': float(t[0]) if n else np.nan,
        'timing__last_event_to_end_seconds': float(duration - t[-1]) if n else np.nan,
        'timing__events_per_active_second': ratio(n - 1, span) if n else np.nan,
        'timing__active_hours': len(set(hours)),
        'timing__active_minutes': len(set(np.floor(t / 60))),
        'timing__hour_entropy': entropy(hours),
    })
    for start in [0, 6, 12, 18]:
        out[f'timing__hour_{start:02d}_{start + 6:02d}_share'] = ratio(
            int(((hours >= start) & (hours < start + 6)).sum()), n
        )
    angles = (
        ev.event_ts.dt.hour * 3600 + ev.event_ts.dt.minute * 60 + ev.event_ts.dt.second
    ).to_numpy() * (2 * np.pi / 86400)
    for name in ['sin', 'cos']:
        out[f'timing__time_of_day_{name}_mean'] = float(getattr(np, name)(angles).mean()) if n else np.nan
    for width in [10, 60, 300]:
        maximum = int((np.searchsorted(t, t + width, side='left') - np.arange(n)).max()) if n else 0
        out[f'timing__rolling_{width}s_max_count'] = maximum
        out[f'timing__rolling_{width}s_max_share'] = ratio(maximum, n)

    # Новая сессия начинается после паузы строго больше 30 минут.
    starts = np.r_[0, np.flatnonzero(gaps > 1800) + 1] if n else np.array([], dtype=int)
    ends = np.r_[starts[1:], n] if n else np.array([], dtype=int)
    sizes = ends - starts
    spans = t[ends - 1] - t[starts] if n else []
    out['sessions__count'] = len(starts)
    add_stats(out, 'sessions__event_count', sizes)
    add_stats(out, 'sessions__duration_seconds', spans)
    out['sessions__singleton_share'] = ratio(int((sizes == 1).sum()), len(sizes))
    out['sessions__largest_session_event_share'] = ratio(int(sizes.max()) if n else 0, n)
    return out


In [9]:
def diversity_features(ev):
    out = {}
    for column in ['item_id', 'item_category', 'item_location', 'seller_type']:
        add_diversity(out, f'diversity__{column}', ev[column].tolist())
    views = ev.loc[ev.event_name.eq('item_view')]
    add_diversity(out, 'diversity__viewed_item', views.item_id.tolist())
    viewed = set(views.item_id) - {''}
    for label, mask in [('contact', ev.event_name.isin(CONTACTS)),
                        ('favorite', ev.event_name.eq('favorite_add')),
                        ('photo', ev.event_name.eq('photo_swipe'))]:
        items = set(ev.loc[mask, 'item_id']) - {''}
        out[f'diversity__{label}_item_nunique'] = len(items)
        out[f'diversity__viewed_items_with_{label}_share'] = ratio(len(viewed & items), len(viewed))
    sellers = [value for value in ev.seller_type if value]
    for seller in ['private', 'pro']:
        out[f'diversity__seller_{seller}_share'] = ratio(sellers.count(seller), len(sellers))
    return out


Для переходов использую только соседние события с однозначным порядком. Если несколько событий имеют одинаковое время, связи через них исключаю. Первый и последний тип события в таком случае получают категорию `ambiguous`.


In [10]:
def sequence_features(row, ev):
    names = ev.event_name.to_numpy(dtype=str)
    n = len(ev)
    t = (ev.event_ts - row.window_start_ts).dt.total_seconds().to_numpy()
    counts = Counter(t)
    singleton = np.array([counts[value] == 1 for value in t], dtype=bool)
    # У событий с одинаковым timestamp неизвестен порядок.
    edge = singleton[:-1] & singleton[1:] & (np.diff(t) > 0)
    pairs = list(zip(names[:-1][edge], names[1:][edge]))
    transitions = Counter(pairs)
    total = int(edge.sum())
    out = {
        'sequence__ordered_pair_count': total,
        'sequence__ordered_pair_fraction': ratio(total, max(n - 1, 0)),
        'sequence__transition_entropy': entropy(pairs),
        'sequence__same_type_share': ratio(sum(v for (a, b), v in transitions.items() if a == b), total),
    }
    for name in EVENTS:
        for left in ['item_view', 'search_results_view']:
            out[f'sequence__{left}_to_{name}_share'] = ratio(transitions[left, name], total)
    for label, index in [('first', 0), ('last', -1)]:
        out[f'sequence__{label}_event'] = (
            str(names[index]) if n and singleton[index] else ('ambiguous' if n else 'missing')
        )
    longest = run = int(n > 0)
    for i in range(1, n):
        run = run + 1 if edge[i - 1] and names[i] == names[i - 1] else 1
        longest = max(longest, run)
    out['sequence__same_type_max_run'] = longest
    for field in ['item_id', 'item_category', 'item_location']:
        values = ev[field].to_numpy(dtype=str)
        valid = edge & (values[:-1] != '') & (values[1:] != '')
        out[f'sequence__{field}_comparable_pairs'] = int(valid.sum())
        out[f'sequence__{field}_switch_share'] = ratio(
            int(((values[:-1] != values[1:]) & valid).sum()), int(valid.sum())
        )
    return out


In [11]:
def platform_features(ev):
    out = {}
    for field, choices in [('platform_norm', PLATFORMS), ('browser', BROWSERS), ('os', OS_NAMES)]:
        values = ev[field].tolist()
        counts = Counter(values)
        out[f'platform_ua__{field}_mode'] = mode(values)
        out[f'platform_ua__{field}_nunique'] = len(set(values))
        for value in choices:
            out[f'platform_ua__{field}_{value}_share'] = ratio(counts[value], len(ev))
    out['platform_ua__ua_nunique'] = len(set(ev.user_agent) - {''})
    out['platform_ua__automation_hint_share'] = ratio(sum(ev.ua_automation_hint), len(ev))
    return out


In [12]:
def pointer_features(row, ev):
    out = {}
    n = len(ev)
    has_pointer = ev.pointer_x.notna() & ev.pointer_y.notna()
    platforms = ev.platform_norm.to_numpy(dtype=str)
    for label, eligible in [('all', np.ones(n, dtype=bool)), ('web', platforms == 'web'),
                            ('mobile', np.isin(platforms, ['android', 'ios']))]:
        out[f'missingness__pointer_{label}_observed_share'] = ratio(
            int((has_pointer & eligible).sum()), int(np.sum(eligible))
        )
    for field in ['item_category', 'item_location', 'seller_type']:
        eligible = ~ev.event_name.isin(['login', 'captcha_shown'])
        if field == 'seller_type':
            eligible &= ev.event_name.ne('search_results_view')
        out[f'missingness__{field}_missing_share'] = ratio(
            int((ev[field].eq('') & eligible).sum()), int(eligible.sum())
        )
    for event_name in ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add']:
        eligible = ev.event_name.eq(event_name) & ev.platform_norm.eq('web')
        out[f'missingness__pointer_web_{event_name}_observed_share'] = ratio(
            int((has_pointer & eligible).sum()), int(eligible.sum())
        )
    pointer = ev.loc[has_pointer & ev.platform_norm.eq('web')]
    pairs = list(zip(pointer.pointer_x, pointer.pointer_y))
    out.update({
        'pointer__web_observation_count': len(pairs),
        'pointer__web_position_nunique': len(set(pairs)),
        'pointer__web_position_unique_fraction': ratio(len(set(pairs)), len(pairs)),
        'pointer__web_position_top_share': ratio(max(Counter(pairs).values(), default=0), len(pairs)),
    })
    for axis in ['x', 'y']:
        add_stats(out, f'pointer__web_{axis}', pointer[f'pointer_{axis}'].to_numpy())
    time_counts = Counter((ev.event_ts - row.window_start_ts).dt.total_seconds())
    rows = list(pointer.itertuples(index=False))
    distances = []
    for left, right in zip(rows, rows[1:]):
        lt = (left.event_ts - row.window_start_ts).total_seconds()
        rt = (right.event_ts - row.window_start_ts).total_seconds()
        if rt > lt and time_counts[lt] == time_counts[rt] == 1:
            distances.append(float(np.hypot(right.pointer_x - left.pointer_x, right.pointer_y - left.pointer_y)))
    out['pointer__web_comparable_pair_count'] = len(distances)
    add_stats(out, 'pointer__web_displacement', distances)
    out['pointer__web_zero_displacement_share'] = ratio(sum(x == 0 for x in distances), len(distances))
    return out


Отбор групп проводился на временной валидации. Убрал отдельный блок характеристик поисковых запросов и страниц выдачи; количество поисковых действий и их переходы осталось. Также исключил признаки, которые были константами на обучающих частях. Состав признаков ниже фиксирован.

Сохраняю промежуточные таблицы в `solution_features/`, чтобы их можно было посмотреть отдельно от обучения.


In [13]:
# Эти константы были исключены на обучающих частях при выборе признаков.
DROP_BASE = ['platform_ua__platform_norm_nunique', 'platform_ua__platform_norm_other_share',
             'platform_ua__platform_norm_missing_share', 'platform_ua__browser_edge_share',
             'platform_ua__browser_other_share', 'platform_ua__browser_missing_share',
             'platform_ua__os_missing_share']
CAT_COLUMNS = ['sequence__first_event', 'sequence__last_event',
               'platform_ua__platform_norm_mode', 'platform_ua__browser_mode', 'platform_ua__os_mode']

groups = dict(tuple(events.groupby('cookie_id', sort=False)))
empty_events = events.iloc[:0]
records = []
for i, row in enumerate(meta.itertuples(index=False), 1):
    ev = groups.get(row.cookie_id, empty_events)
    features = {'cookie_id': row.cookie_id}
    features.update(activity_features(row, ev))
    features.update(timing_features(row, ev))
    features.update(diversity_features(ev))
    features.update(sequence_features(row, ev))
    features.update(platform_features(ev))
    features.update(pointer_features(row, ev))
    records.append(features)
    if i % 4000 == 0:
        print(f'Признаки: {i} / {len(meta)}')
base_features = pd.DataFrame(records).set_index('cookie_id').drop(columns=DROP_BASE)
assert base_features.shape == (len(meta), 199)
base_features.to_csv(FEATURE_DIR / 'base_features.csv')
base_features = pd.read_csv(
    FEATURE_DIR / 'base_features.csv', dtype={c: str for c in CAT_COLUMNS}
).set_index('cookie_id')
BASE_COLUMNS = base_features.columns.tolist()
base_features.shape


Признаки: 4000 / 16000
Признаки: 8000 / 16000
Признаки: 12000 / 16000
Признаки: 16000 / 16000


(16000, 199)

## 3. Общие объявления

Из событий `item_view` получаю множество объявлений каждой cookie. Затем считаю частоты этих объявлений у других cookie, долю совпавших объявлений, максимальное пересечение наборов и сходство в смысле число общих объявлений, делённое на число объявлений в объединённом наборе. Повторные просмотры одного объявления одной cookie дают один голос; собственный вклад исключаю.

Для окна с окончанием в `t` использую два контекста:

- предыдущие семь суток: окна с окончанием в `[t − 7 суток, t)`;
- текущие сутки: другие окна с окончанием ровно в `t`.


Если фон отсутствует, частоты остаются `NaN`; если фон есть, но совпадений нет, получаем ноль. Сходство считаю среди соседей хотя бы с двумя общими объявлениями, чтобы одно популярное объявление не давало высокое сходство.


In [14]:
def build_item_features(meta, events):
    """Пересечения просмотренных объявлений с другими куки к концу суток."""
    assert "target" not in meta and "target" not in events
    assert meta.cookie_id.is_unique and meta.cookie_id.notna().all()
    day = pd.Timedelta(days=1)
    assert (meta.window_end_ts - meta.window_start_ts).eq(day).all()
    assert (meta.window_start_ts - meta.window_start_ts.iloc[0]).mod(day).eq(pd.Timedelta(0)).all()
    windows = meta.set_index("cookie_id")
    assert events.cookie_id.isin(windows.index).all()
    assert events.event_ts.ge(events.cookie_id.map(windows.window_start_ts)).all()
    assert events.event_ts.lt(events.cookie_id.map(windows.window_end_ts)).all()

    # Повторные просмотры одного объявления дают только один голос от куки.
    positions = {cookie: i for i, cookie in enumerate(meta.cookie_id)}
    items = [set() for _ in range(len(meta))]
    views = events.loc[events.event_name.eq("item_view"), ["cookie_id", "item_id"]]
    for cookie, item in views.itertuples(index=False, name=None):
        if pd.notna(item) and str(item).strip():
            items[positions[cookie]].add(str(item).strip())

    frequency_stats = [
        "mean_peer_count", "max_peer_count", "mean_peer_fraction",
        "max_peer_fraction", "share_seen", "share_seen_ge3",
    ]
    pair_stats = [
        "peer_share_shared_ge1", "peer_share_shared_ge2", "max_shared_value_count",
        "max_own_overlap_fraction", "max_jaccard_shared_ge2",
    ]
    columns = []
    for context in ["history7", "same_day"]:
        names = (["ref_available"] if context == "history7" else []) + frequency_stats + pair_stats
        columns.extend(f"cross_cookie__items_{context}__{name}" for name in names)
    rows = [dict.fromkeys(columns, np.nan) for _ in range(len(meta))]

    ends = meta.window_end_ts
    for end in sorted(ends.unique()):
        same_day = np.flatnonzero(ends.eq(end).to_numpy())
        history = np.flatnonzero((ends.ge(end - 7 * day) & ends.lt(end)).to_numpy())
        for context, reference in [("history7", history), ("same_day", same_day)]:
            # Объявление -> множество других куки; пустые наборы не входят в знаменатель.
            owners = defaultdict(set)
            sizes = {}
            for j in reference:
                if items[j]:
                    sizes[j] = len(items[j])
                    for item in items[j]:
                        owners[item].add(j)

            prefix = f"cross_cookie__items_{context}__"
            for i in same_day:
                own = items[i]
                n_peers = len(sizes) - int(i in sizes)
                row = rows[i]
                if context == "history7":
                    row[prefix + "ref_available"] = float(n_peers > 0)
                if not n_peers:
                    continue
                if own:
                    counts = np.asarray([
                        len(owners.get(item, ())) - int(i in owners.get(item, ()))
                        for item in sorted(own)
                    ], dtype=float)
                    values = [counts.mean(), counts.max(), counts.mean() / n_peers,
                              counts.max() / n_peers, (counts > 0).mean(), (counts >= 3).mean()]
                    for name, value in zip(frequency_stats, values):
                        row[prefix + name] = float(value)

                shared = Counter()
                for item in sorted(own):
                    shared.update(owners.get(item, ()))
                shared.pop(i, None)  # Собственная куки не считается совпадением.
                maximum = max(shared.values(), default=0)
                row[prefix + "peer_share_shared_ge1"] = len(shared) / n_peers
                row[prefix + "peer_share_shared_ge2"] = sum(n >= 2 for n in shared.values()) / n_peers
                row[prefix + "max_shared_value_count"] = float(maximum)
                row[prefix + "max_own_overlap_fraction"] = maximum / len(own) if own else np.nan
                # Один популярный товар ещё не означает похожий набор просмотренных объявлений.
                if len(own) >= 2 and any(j != i and size >= 2 for j, size in sizes.items()):
                    row[prefix + "max_jaccard_shared_ge2"] = max(
                        (n / (len(own) + sizes[j] - n) for j, n in shared.items() if n >= 2),
                        default=0.0,
                    )

    return pd.DataFrame(rows, columns=columns, dtype=float,
                        index=pd.Index(meta.cookie_id, name="cookie_id"))


In [15]:
item_features = build_item_features(meta, events)
item_features.to_csv(FEATURE_DIR / 'item_features.csv')
item_features = pd.read_csv(FEATURE_DIR / 'item_features.csv').set_index('cookie_id')
features = base_features.join(item_features, validate='one_to_one')
assert features.shape == (len(meta), 222)
assert features.index.tolist() == meta.cookie_id.tolist()
assert not features[CAT_COLUMNS].isna().any().any()
assert not np.isinf(features.drop(columns=CAT_COLUMNS).to_numpy(dtype=float)).any()

pd.Series(features.columns.str.split('__').str[0]).value_counts().rename('Количество признаков').to_frame()


,Количество признаков
timing,41
diversity,33
sequence,33
activity,29
platform_ua,23
cross_cookie,23
pointer,18
sessions,11
missingness,10
metadata,1


## 4. Валидация и baseline

Использую растущий train и непересекающиеся проверочные периоды:

| Обучение | Проверка |
|---|---|
| 6–10 апреля | 11–13 апреля |
| 6–13 апреля | 14–16 апреля |
| 6–16 апреля | 17–19 апреля |

Такое разбиение ближе к предсказанию будущего трафика, чем случайное перемешивание cookie.


In [16]:
periods = [('2026-04-11', '2026-04-14'),
           ('2026-04-14', '2026-04-17'),
           ('2026-04-17', '2026-04-20')]
splits = []
split_rows = []
for start, end in periods:
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    fit = train.loc[train.window_end_ts.le(start)]
    valid = train.loc[train.window_start_ts.ge(start) & train.window_end_ts.le(end)]
    assert set(fit.cookie_id).isdisjoint(valid.cookie_id)
    label = f'{start.day}–{end.day - 1} апреля'
    splits.append((label, fit.cookie_id.tolist(), valid.cookie_id.tolist()))
    split_rows.append({'Проверка': label, 'Train': len(fit), 'Ботов в train': int(fit.target.sum()),
                       'Validation': len(valid), 'Ботов в validation': int(valid.target.sum())})
y = train.set_index('cookie_id').target
pd.DataFrame(split_rows)


,Проверка,Train,Ботов в train,Validation,Ботов в validation
0,11–13 апреля,4217,345,2556,199
1,14–16 апреля,6773,544,2367,195
2,17–19 апреля,9140,739,1951,160


Основная метрика — максимальный precision среди порогов с recall не ниже 70%. Одинаковые score рассматриваются вместе. Дополнительно сохраняю Average Precision (AP) и ROC-AUC.


In [17]:
def precision_at_recall(y_true, score, min_recall=0.70):
    precision, recall, thresholds = precision_recall_curve(y_true, score)
    # sklearn перебирает уникальные пороги: одинаковые score остаются одной группой.
    eligible = recall[:-1] >= min_recall
    return float(precision[:-1][eligible].max())


# При одинаковых оценках нельзя выбрать только один из двух объектов.
assert precision_at_recall([1, 0], [0.5, 0.5]) == 0.5


В обоих вариантах использую одинаковый ансамбль и параметры: CatBoost с глубиной 5 и LightGBM с 31 листом, по 1000 деревьев. Для каждого алгоритма усредняю seed `42, 17, 123`, затем смешиваю результаты: `0.75 × CatBoost + 0.25 × LightGBM`.

Категории CatBoost получает строками. Для LightGBM словарь строится только на обучающей части; неизвестные категории превращаются в пропуски. Веса классов и early stopping не использую.


In [18]:
CB_PARAMS = dict(iterations=1000, depth=5, learning_rate=0.05, l2_leaf_reg=5,
                 loss_function='Logloss', task_type='CPU', thread_count=2,
                 verbose=False, allow_writing_files=False, use_best_model=False)
LGB_PARAMS = dict(objective='binary', n_estimators=1000, learning_rate=0.05,
                  num_leaves=31, max_depth=-1, min_child_samples=50,
                  reg_lambda=0, reg_alpha=0.0, subsample=0.9, subsample_freq=1,
                  colsample_bytree=0.9, deterministic=True, force_col_wise=True,
                  n_jobs=2, verbosity=-1, zero_as_missing=False, device_type='cpu')


def fit_predict(x_train, y_train, x_predict):
    # Словарь LightGBM строится только на обучающей части.
    lgb_train, lgb_predict = x_train.copy(), x_predict.copy()
    for column in CAT_COLUMNS:
        categories = sorted(x_train[column].dropna().unique().tolist())
        dtype = pd.CategoricalDtype(categories=categories, ordered=False)
        lgb_train[column] = lgb_train[column].astype(dtype)
        lgb_predict[column] = lgb_predict[column].astype(dtype)

    cb_scores, lgb_scores = [], []
    for seed in SEEDS:
        cb = CatBoostClassifier(**CB_PARAMS, random_seed=seed)
        cb.fit(x_train, y_train, cat_features=CAT_COLUMNS)
        cb_scores.append(cb.predict_proba(x_predict)[:, 1])

        lgb = LGBMClassifier(**LGB_PARAMS, random_state=seed, bagging_seed=seed,
                            feature_fraction_seed=seed, data_random_seed=seed)
        lgb.fit(lgb_train, y_train, categorical_feature=CAT_COLUMNS)
        lgb_scores.append(lgb.predict_proba(lgb_predict)[:, 1])
        print(f'  seed {seed}: готово', flush=True)
    return 0.75 * np.mean(cb_scores, axis=0) + 0.25 * np.mean(lgb_scores, axis=0)


Baseline этого сравнения — тот же ансамбль на 199 признаках самой cookie. Улучшенный вариант добавляет 23 признака общих объявлений. Ниже оба варианта обучаются заново на каждом разбиении.


In [19]:
results = []
for label, fit_ids, valid_ids in splits:
    for name, columns in [('Базовые признаки', BASE_COLUMNS),
                          ('Базовые + общие объявления', features.columns.tolist())]:
        print(f'{label}, {name}', flush=True)
        score = fit_predict(features.loc[fit_ids, columns], y.loc[fit_ids],
                            features.loc[valid_ids, columns])
        results.append({'Проверка': label, 'Признаки': name,
                        'P@R≥70%': precision_at_recall(y.loc[valid_ids], score),
                        'AP': average_precision_score(y.loc[valid_ids], score),
                        'ROC-AUC': roc_auc_score(y.loc[valid_ids], score)})
        print(f"  P@R≥70%: {results[-1]['P@R≥70%']:.2%}", flush=True)
validation_results = pd.DataFrame(results)
validation_results.to_csv('validation_results.csv', index=False)
validation_results.pivot(index='Признаки', columns='Проверка', values='P@R≥70%').round(4)


11–13 апреля, Базовые признаки
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 78.21%
11–13 апреля, Базовые + общие объявления
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 82.94%
14–16 апреля, Базовые признаки
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 77.84%
14–16 апреля, Базовые + общие объявления
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 87.82%
17–19 апреля, Базовые признаки
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 73.20%
17–19 апреля, Базовые + общие объявления
  seed 42: готово
  seed 17: готово
  seed 123: готово
  P@R≥70%: 86.26%


Проверка,11–13 апреля,14–16 апреля,17–19 апреля
Признаки,,,
Базовые + общие объявления,0.8294,0.8782,0.8626
Базовые признаки,0.7821,0.7784,0.7320


## 5. Что ещё проверял

Начал с простых счётчиков, затем сравнивал группы признаков и их исключение. Проверял параметры CatBoost и LightGBM, веса классов, регуляризацию и веса ансамбля. Сохранил смесь 75/25 без весов классов.

Увеличение числа seed до семи, дополнительное сокращение признаков, отдельные модели для Android(пробовал, потому что на андроид низкий recall) и новые признаки ритма не дали устойчивого выигрыша. TF-IDF последовательностей с логистической регрессией и небольшая GRU также не улучшили качество на первых двух периодах.

Из кросс-признаков сравнивал общие объявления, запросы и User-Agent. Лучший результат дали объявления: P@R≥70% вырос с **78,21% / 77,84% / 73,20%** до **82,94% / 87,82% / 86,26%**. Таблица выше пересчитывает это сравнение.



## 6. Финальное обучение и submission

Обучаю выбранный ансамбль на всём размеченном train.

In [20]:
# Сохраняем итоговые таблицы: их можно отдельно посмотреть и использовать для повторного обучения.
features.loc[train.cookie_id].to_csv(FEATURE_DIR / 'train_features.csv')
features.loc[test.cookie_id].to_csv(FEATURE_DIR / 'test_features.csv')
x_train = pd.read_csv(FEATURE_DIR / 'train_features.csv', dtype={c: str for c in CAT_COLUMNS}).set_index('cookie_id')
x_test = pd.read_csv(FEATURE_DIR / 'test_features.csv', dtype={c: str for c in CAT_COLUMNS}).set_index('cookie_id')
assert x_train.index.equals(y.index)
print(f'Финальное обучение: {len(x_train)} cookie, {x_train.shape[1]} признака', flush=True)
test_score = fit_predict(x_train, y, x_test)


Финальное обучение: 11091 cookie, 222 признака
  seed 42: готово
  seed 17: готово
  seed 123: готово


In [21]:
sample = pd.read_csv('sample_submission.csv', dtype={'cookie_id': str})
predictions = pd.Series(test_score, index=x_test.index, name='score')
# Порядок строк test и sample различается, поэтому сопоставляем оценки по cookie_id.
submission = sample[['cookie_id']].copy()
submission['score'] = submission.cookie_id.map(predictions)
assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert set(submission.cookie_id) == set(test.cookie_id)
assert submission.columns.tolist() == ['cookie_id', 'score']
assert np.isfinite(submission.score).all() and submission.score.between(0, 1).all()
submission.to_csv('submission.csv', index=False, float_format='%.17g')
print(f'Сохранён submission.csv: {len(submission)} строк')
submission.head()


Сохранён submission.csv: 4909 строк


,cookie_id,score
0,ck_99a4e5ef02d89493,0.015211
1,ck_54d463f7fd89ec3d,0.013744
2,ck_0fbbc7a6af300368,0.001756
3,ck_8fd4937eadd64fb6,0.001587
4,ck_dbb4bd4eda97255d,0.004114
